In [2]:
import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path
from tqdm import tqdm
from scipy.signal import butter, filtfilt, resample_poly

# ----------------------- 配置 -----------------------
ROOT = Path(r"D:\Project_Github\audio_click_mil")

AUDIO_DIR = ROOT / "data" / "original_audio"
BAG_CSV = ROOT / "processed_data" / "balanced_bag_labels.csv"

OUTPUT_DIR = ROOT / "processed_data" / "balanced_bags"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# 新增：instance级标签输出路径
INSTANCE_LABEL_CSV = ROOT / "processed_data" / "instance_labels.csv"

EXCLUDE_FILES = {12}
TARGET_SR = 200000          # 目标采样率 200kHz
LOW_CUTOFF = 5000           # 带通低 cutoff
HIGH_CUTOFF = 200000        # 带通高 cutoff

# 读取三个ground truth csv用于判断instance是否为正
GT_PATHS = [
    ROOT / "data" / "ClickTrains.csv",
    ROOT / "data" / "BurstPulseTrains.csv",
    ROOT / "data" / "BuzzTrains.csv"
]

# ----------------------- 工具函数 -----------------------
def parse_file_num(filename: str) -> int:
    return int(Path(filename).stem[-2:])


def bandpass_filter(signal: np.ndarray, sr: int, low=5000, high=200000, order=6):
    """5kHz - 200kHz 带通滤波"""
    nyquist = 0.5 * sr
    low_norm = low / nyquist
    high_norm = high / nyquist
    b, a = butter(order, [low_norm, high_norm], btype='band')
    return filtfilt(b, a, signal)


def resample_audio(signal: np.ndarray, orig_sr: int, target_sr: int):
    """降采样到目标采样率"""
    if orig_sr == target_sr:
        return signal
    return resample_poly(signal, target_sr, orig_sr)


def is_instance_positive(file_num: int, instance_start_sec: float, gt_dfs: list) -> int:
    """判断该1秒instance是否与任何pulse train有重叠（任意重叠即为正）"""
    instance_end_sec = instance_start_sec + 1.0
    for df in gt_dfs:
        gt = df[df['Ori_file_num(No.)'] == file_num]
        for _, row in gt.iterrows():
            train_start = row['Train_start(s)']
            train_end = row['Train_end(s)']
            if max(instance_start_sec, train_start) < min(instance_end_sec, train_end):
                return 1
    return 0


# ----------------------- 主流程 -----------------------
# ... 前面工具函数部分保持不变 ...

# ----------------------- 主流程 -----------------------
def build_instances():
    print("读取 balanced_bag_labels.csv ...")
    bag_df = pd.read_csv(BAG_CSV)

    print("加载ground truth文件用于instance标签...")
    gt_dfs = []
    for p in GT_PATHS:
        if p.exists():
            gt_dfs.append(pd.read_csv(p))
        else:
            print(f"警告: GT文件不存在 {p}")

    instance_records = []

    for file_num, file_group in tqdm(bag_df.groupby("file_num"), desc="Processing files"):
        if file_num in EXCLUDE_FILES:
            continue

        audio_filename = file_group["audio_path"].iloc[0].split("\\")[-1]
        audio_path = AUDIO_DIR / audio_filename

        if not audio_path.exists():
            print(f"警告: 音频文件不存在 {audio_path}")
            continue

        info = sf.info(str(audio_path))
        orig_sr = info.samplerate

        # ================== 新增：Per Audiofile 归一化模块 ==================
        print(f"\n正在扫描文件最大幅值以进行归一化: {audio_filename}")
        global_max = 0
        # 使用较小的 blocksize 快速流式扫描全文件，找出绝对值最大点
        for block in sf.blocks(str(audio_path), blocksize=orig_sr * 600): # 每次读10分钟扫描
            # 如果是多通道，取均值
            if block.ndim > 1:
                block = np.mean(block, axis=1)
            curr_max = np.max(np.abs(block))
            if curr_max > global_max:
                global_max = curr_max
        
        # 防止除以零
        norm_factor = global_max if global_max > 0 else 1.0
        print(f"文件 {audio_filename} 最大幅值为: {global_max:.4f}, 将以此进行归一化。")
        # =================================================================

        # 按 bag 处理
        for _, row in file_group.iterrows():
            bag_idx = int(row["bag_idx"])
            bag_label = int(row["bag_label"])
            bag_start_sec = float(row["bag_start_sec"])

            start_frame = int(bag_start_sec * orig_sr)
            frames_to_read = int(60 * orig_sr)
            
            try:
                segment_raw, _ = sf.read(str(audio_path), start=start_frame, frames=frames_to_read, dtype='float32')
                
                if len(segment_raw) < frames_to_read:
                    pad_len = frames_to_read - len(segment_raw)
                    segment_raw = np.pad(segment_raw, (0, pad_len), mode='constant')
                
                if segment_raw.ndim > 1:
                    segment_raw = np.mean(segment_raw, axis=1)

                # --- 核心操作：应用全局归一化到 [-1, 1] ---
                segment_raw = segment_raw / norm_factor

                # 滤波（此时处理的是归一化后的数据）
                segment_filtered = bandpass_filter(segment_raw, orig_sr, LOW_CUTOFF, HIGH_CUTOFF)

                # 降采样
                segment_resampled = resample_audio(segment_filtered, orig_sr, TARGET_SR)
                
                target_len = 60 * TARGET_SR
                if len(segment_resampled) != target_len:
                    if len(segment_resampled) > target_len:
                        segment_resampled = segment_resampled[:target_len]
                    else:
                        segment_resampled = np.pad(segment_resampled, (0, target_len - len(segment_resampled)))

                final_bag = segment_resampled.reshape(60, TARGET_SR).astype(np.float32)

                # 保存 .npy
                npy_name = f"file_{file_num:02d}_bag_{bag_idx:03d}_label_{bag_label}.npy"
                npy_path = OUTPUT_DIR / npy_name
                np.save(npy_path, final_bag)

                # 记录标签...
                for inst_idx in range(60):
                    instance_start_sec = bag_start_sec + inst_idx * 1.0
                    inst_label = is_instance_positive(file_num, instance_start_sec, gt_dfs)
                    instance_records.append({
                        "file_num": file_num,
                        "bag_idx": bag_idx,
                        "instance_idx": inst_idx,
                        "bag_label": bag_label,
                        "instance_label": inst_label,
                        "instance_start_sec": round(instance_start_sec, 4),
                        "npy_file": npy_name
                    })

            except Exception as e:
                print(f"处理 Bag {bag_idx} 失败: {e}")
                continue



    # 保存instance级标签CSV
    instance_df = pd.DataFrame(instance_records)
    instance_df.to_csv(INSTANCE_LABEL_CSV, index=False)
    print(f"\nInstance级标签已保存至: {INSTANCE_LABEL_CSV}")
    print(f"总instance数量: {len(instance_df)}")
    print(f"正instance比例: {instance_df['instance_label'].mean():.4f} ({instance_df['instance_label'].sum()}/{len(instance_df)})")

    print(f"\n所有 bag 的实例 .npy 文件生成完成！")
    print(f"保存目录: {OUTPUT_DIR}")
    print(f"每个 .npy 形状: (60, 200000)")


if __name__ == "__main__":
    build_instances()

读取 balanced_bag_labels.csv ...
加载ground truth文件用于instance标签...


Processing files:   0%|          | 0/34 [00:00<?, ?it/s]


正在扫描文件最大幅值以进行归一化: Ori_Recording_01.wav
文件 Ori_Recording_01.wav 最大幅值为: 0.8429, 将以此进行归一化。


Processing files:   3%|▎         | 1/34 [00:24<13:24, 24.37s/it]


正在扫描文件最大幅值以进行归一化: Ori_Recording_02.wav
文件 Ori_Recording_02.wav 最大幅值为: 0.3159, 将以此进行归一化。


...


正在扫描文件最大幅值以进行归一化: Ori_Recording_34.wav
文件 Ori_Recording_34.wav 最大幅值为: 0.3789, 将以此进行归一化。


Processing files:  97%|█████████▋| 33/34 [16:28<00:20, 20.83s/it]


正在扫描文件最大幅值以进行归一化: Ori_Recording_35.wav
文件 Ori_Recording_35.wav 最大幅值为: 0.9900, 将以此进行归一化。


Processing files: 100%|██████████| 34/34 [16:43<00:00, 29.51s/it]



Instance级标签已保存至: D:\Project_Github\audio_click_mil\processed_data\instance_labels.csv
总instance数量: 38160
正instance比例: 0.0480 (1832/38160)

所有 bag 的实例 .npy 文件生成完成！
保存目录: D:\Project_Github\audio_click_mil\processed_data\balanced_bags
每个 .npy 形状: (60, 200000)
